### Notebook 范围

### 目的
比较 O3、EP100 W1+W2、U60N10、U60N50 的 ensemble spread onset。

### 科学问题
动力变量 spread 是否早于 O3 spread？

### 方法
spread(t)=成员标准差；standardized spread=(spread(t)-spread(t0))/std；5-day running mean 首次超过自身最大值的 50% 且维持 5 天定义为 onset。

### 输出
outputs/figures/04_spread_timing 和 outputs/tables/04_spread_timing。

### 解读
EP/U onset lead day 小于 O3 onset 支持 dynamics-first uncertainty pathway。

### 注意
如果 case 初始化较晚，divergence 可能已在初始化前发生，因此 onset 不应解释为 long-lead precursor。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 spread onset lead-day summary

### 目的
计算每个变量的 spread onset，并用距离初始化的 lead day 作横轴。

### 科学问题
去掉 DOY 后，不同初始化月份能否公平比较 spread 出现的相对时间？

### 方法
EP100 W1+W2 使用 wave1+wave2 的 mean(-ep2)，100 hPa，40-80N；O3 使用 partial column；U 使用 60N 10/50 hPa。

### 输出
04_spread_onset_timing_summary_all_cases_v2.png/pdf 和 CSV。

### 解读
越靠左表示越早出现成员分歧；EP/U 在 O3 左侧说明 dynamical spread 领先臭氧 spread。

### 注意
onset 阈值是经验诊断，逐渐增长或初始已有 spread 时会有不确定性。


In [ ]:
fig_dir = figure_dir("04_spread_timing")
tab_dir = table_dir("04_spread_timing")
inv = discover_hindcast_cases()
rows = []
for case in inv.loc[inv["has_partial_O3"], "case"]:
    o3, _ = load_hindcast_o3(case)
    if o3 is not None:
        onset = compute_spread_onset(o3)
        rows.append({"case": case, "variable": "O3", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
    ep1, _ = load_epflux(case, "wave1")
    ep2, _ = load_epflux(case, "wave2")
    if ep1 is not None and ep2 is not None:
        ep = coslat_weighted_mean(-(ep1.sel(plev=10000, method="nearest") + ep2.sel(plev=10000, method="nearest")), LAT_EP)
        onset = compute_spread_onset(ep)
        rows.append({"case": case, "variable": "EP100_W1W2", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
    for plev in [10, 50]:
        u, _ = compute_u60(case, plev)
        if u is not None:
            onset = compute_spread_onset(u)
            rows.append({"case": case, "variable": f"U60N{plev}", **{k:v for k,v in onset.items() if not isinstance(v, np.ndarray)}})
spread = pd.DataFrame(rows)
spread_csv = tab_dir / "04_spread_onset_timing_summary_all_cases.csv"
spread.to_csv(spread_csv, index=False)
spread.to_csv(TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv", index=False)
fig, ax = plt.subplots(figsize=(11.5, max(5, 0.36 * spread["case"].nunique() + 2 if not spread.empty else 5)), constrained_layout=True)
if spread.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No spread onset diagnostics", ha="center", va="center")
else:
    cases = sorted(spread["case"].unique())
    ymap = {c:i for i, c in enumerate(cases)}
    markers = {"EP100_W1W2":"^", "U60N10":"o", "U60N50":"s", "O3":"D"}
    colors = {"EP100_W1W2":"tab:purple", "U60N10":"tab:orange", "U60N50":"tab:green", "O3":"tab:blue"}
    labels = {"EP100_W1W2":"EP100 W1+W2 spread", "U60N10":"U60N10 spread", "U60N50":"U60N50 spread", "O3":"O3 spread"}
    for var, sub in spread.groupby("variable"):
        ax.scatter(sub["onset_lead_day"], [ymap[c] for c in sub["case"]], marker=markers.get(var, "o"), s=85, color=colors.get(var), label=labels.get(var, var), edgecolor="black")
    ax.set_yticks(range(len(cases)), cases)
    ax.set_xlabel("Spread onset lead day from initialization")
    ax.set_title("Spread onset timing\nspread(t)=ensemble std; onset=5-day running standardized spread > 50% max for 5 days")
    ax.legend(ncol=2, fontsize=9, title="Variable and definition")
    ax.grid(True, axis="x", alpha=0.25)
savefig(fig, "04_spread_onset_timing_summary_all_cases_v2", fig_dir=fig_dir, notebook="04_spread_timing_multicase.ipynb", scientific_question="EP/vortex spread 是否早于 O3 spread？", variables_windows="O3 partial column; EP100 W1+W2 mean(-ep2) 100 hPa 40-80N; U60N10/U60N50; x=lead day", interpretation="EP/U marker 在 O3 左侧支持 dynamics-first uncertainty propagation。", caveat="阈值 onset 适合比较相对 timing，不是严格物理突变点。", csv_table=spread_csv)
plt.show()
write_figure_guide()
